# 1. Environment Setup

## 1.3. Mount Google Drive (for Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: Create a symbolic link or change directory to your project folder
# Example: !ln -s "/content/drive/My Drive/rl_project" "/content/rl_project"
# %cd /content/rl_project

## 1.4. Install Required Libraries

In [ ]:
!pip install tensorflow
!pip install gymnasium[atari,accept-rom-license]
!pip install numpy
!pip install matplotlib

# 2. DQN Agent Implementation

## 2.1. Imports and Hyperparameters

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Input
from tensorflow.keras.optimizers import Adam
import gymnasium as gym
import random
from collections import deque
import matplotlib.pyplot as plt

# Environment parameters
STATE_SHAPE = (84, 84, 4)  # Input shape for the CNN (after preprocessing and frame stacking)
# ACTION_SIZE will be determined from the environment

# Hyperparameters
LEARNING_RATE = 0.00025
GAMMA = 0.99  # Discount factor
EPSILON_START = 1.0  # Starting exploration rate
EPSILON_END = 0.1    # Minimum exploration rate
EPSILON_DECAY_FRAMES = 1000000  # Frames over which to decay epsilon

REPLAY_BUFFER_SIZE = 100000  # Max size of the replay buffer (reduced for faster example, typical is 1M)
BATCH_SIZE = 32

TARGET_NETWORK_UPDATE_FREQ = 10000  # How often to update the target network (in steps)
TRAINING_START_STEP = 10000 # Start training after this many steps (fill buffer)

# Preprocessing parameters
SKIP_FRAMES = 4
STACK_SIZE = 4 # Number of frames to stack

## 2.2. Replay Buffer

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def remember(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

## 2.3. DQN Neural Network Model

In [ ]:
def build_dqn_model(input_shape, num_actions):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(32, kernel_size=8, strides=4, activation='relu', padding='same'),
        Conv2D(64, kernel_size=4, strides=2, activation='relu', padding='same'),
        Conv2D(64, kernel_size=3, strides=1, activation='relu', padding='same'),
        Flatten(),
        Dense(512, activation='relu'),
        Dense(num_actions, activation='linear')  # Q-values
    ])
    return model

## 2.4. DQN Agent

In [ ]:
class DQNAgent:
    def __init__(self, state_shape, action_size):
        self.state_shape = state_shape
        self.action_size = action_size
        self.epsilon = EPSILON_START
        self.epsilon_decay_rate = (EPSILON_START - EPSILON_END) / EPSILON_DECAY_FRAMES

        self.model = build_dqn_model(state_shape, action_size)
        self.target_model = build_dqn_model(state_shape, action_size)
        self.target_model.set_weights(self.model.get_weights()) # Initialize target model
        
        self.optimizer = Adam(learning_rate=LEARNING_RATE, clipnorm=1.0) # Using clipnorm for stability
        self.replay_buffer = ReplayBuffer(REPLAY_BUFFER_SIZE)
        
        # Loss function (Huber loss for stability)
        self.loss_function = tf.keras.losses.Huber()

    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state, done):
        self.replay_buffer.remember(state, action, reward, next_state, done)

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        # Add batch dimension for model prediction
        state_tensor = tf.convert_to_tensor(state)
        state_tensor = tf.expand_dims(state_tensor, 0)
        q_values = self.model(state_tensor, training=False)
        return np.argmax(q_values[0].numpy())

    def replay(self, batch_size):
        if len(self.replay_buffer) < TRAINING_START_STEP or len(self.replay_buffer) < batch_size : # Ensure buffer has enough samples
            return 0 # Return 0 loss if not training

        minibatch = self.replay_buffer.sample(batch_size)
        
        states = np.array([experience[0] for experience in minibatch])
        actions = np.array([experience[1] for experience in minibatch])
        rewards = np.array([experience[2] for experience in minibatch])
        next_states = np.array([experience[3] for experience in minibatch])
        dones = np.array([experience[4] for experience in minibatch])

        # Convert to tensors
        states_tensor = tf.convert_to_tensor(states, dtype=tf.float32)
        next_states_tensor = tf.convert_to_tensor(next_states, dtype=tf.float32)
        rewards_tensor = tf.convert_to_tensor(rewards, dtype=tf.float32)
        actions_tensor = tf.convert_to_tensor(actions, dtype=tf.int32)
        dones_tensor = tf.convert_to_tensor(dones, dtype=tf.bool)

        # Get Q-values for next_states from target_model
        # For DQN, use the main model to select the best action for the next state (Double DQN uses target model here)
        # However, standard DQN evaluates these actions using the target model.
        future_q_values_target = self.target_model(next_states_tensor, training=False)
        
        # Calculate target Q-values: R + gamma * max_a' Q_target(s', a')
        # For terminal states, the target is just the reward
        target_q_values = rewards_tensor + (GAMMA * tf.reduce_max(future_q_values_target, axis=1) * (1 - tf.cast(dones_tensor, tf.float32)))
        
        # Create a mask for the actions taken
        action_masks = tf.one_hot(actions_tensor, self.action_size)

        with tf.GradientTape() as tape:
            # Get Q-values for current states from the main model
            q_values = self.model(states_tensor, training=True)
            # Select the Q-values for the actions that were actually taken
            predicted_q_values = tf.reduce_sum(q_values * action_masks, axis=1)
            # Calculate loss
            loss = self.loss_function(target_q_values, predicted_q_values)

        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.model.trainable_variables))
        
        # Decay epsilon
        if self.epsilon > EPSILON_END:
            self.epsilon -= self.epsilon_decay_rate
        
        return loss.numpy()

    def load(self, name):
        self.model.load_weights(name)
        self.update_target_model() # Ensure target model is also updated

    def save(self, name):
        self.model.save_weights(name)

print("DQN Agent, Model, and Replay Buffer defined.")

# 3. Training the DQN Agent

## 3.1. Frame Preprocessing and Stacking

In [ ]:
from skimage.color import rgb2gray
from skimage.transform import resize
# Preprocessing function based on DeepMind's approach for Atari
def preprocess_frame(frame):
    # Convert to grayscale
    gray_frame = rgb2gray(frame)
    # Resize to 84x84
    resized_frame = resize(gray_frame, (84, 84), anti_aliasing=True)
    # Normalize pixel values (optional, but good practice)
    # resized_frame = resized_frame / 255.0 
    # The CNN in the agent expects values in [0, 255] or [0,1] depending on how it was trained or defined.
    # For now, let's assume raw pixel values after resize, or scale if needed.
    # To match the typical input for Conv2D, we might need to expand dims if it's just (84,84)
    # return np.expand_dims(resized_frame, axis=-1) # (84, 84, 1)
    return resized_frame # (84,84)

# Function to stack frames
def stack_frames(stacked_frames, frame, is_new_episode):
    if is_new_episode:
        # For a new episode, fill the stack with the same initial frame
        processed_frame = preprocess_frame(frame)
        stacked_frames = deque([processed_frame for _ in range(STACK_SIZE)], maxlen=STACK_SIZE)
    else:
        # Append the new frame and remove the oldest
        processed_frame = preprocess_frame(frame)
        stacked_frames.append(processed_frame)
    
    # Stack along the last dimension to get shape (84, 84, STACK_SIZE)
    # The deque stores 2D arrays. We stack them to form a 3D array for the CNN.
    try:
        stacked_state = np.stack(stacked_frames, axis=-1) 
    except ValueError as e:
        print(f"Error stacking frames: {e}")
        print(f"Shapes of frames in deque: {[f.shape for f in stacked_frames]}")
        # Fallback: if stacking fails (e.g. inconsistent shapes somehow), re-initialize
        processed_frame = preprocess_frame(frame) # Re-process current frame
        stacked_frames = deque([processed_frame for _ in range(STACK_SIZE)], maxlen=STACK_SIZE)
        stacked_state = np.stack(stacked_frames, axis=-1)

    return stacked_state, stacked_frames

print("Preprocessing functions defined.")

## 3.2. Training Loop

In [ ]:
# Initialize environment and agent
try:
    env = gym.make("SpaceInvaders-v0", obs_type="rgb") # Ensure we get RGB frames
except Exception as e:
    print(f"Error creating SpaceInvaders-v0: {e}")
    print("Trying SpaceInvaders-v4 as a fallback or ensure ROMs are correctly installed.")
    try:
        env = gym.make("SpaceInvaders-v4", obs_type="rgb")
    except Exception as e_v4:
        print(f"Error creating SpaceInvaders-v4: {e_v4}")
        print("Please ensure Atari ROMs are installed: `pip install gymnasium[atari,accept-rom-license]`")
        raise e_v4 # Re-raise the error if fallback also fails

ACTION_SIZE = env.action_space.n
print(f"Action space size: {ACTION_SIZE}")

# Re-initialize agent with correct action_size if it wasn't set before
# or if the environment creation was deferred.
agent = DQNAgent(STATE_SHAPE, ACTION_SIZE)

# Training parameters
NUM_EPISODES = 500 # Reduced for a quicker example run; typical: 1000s-10000s
MAX_STEPS_PER_EPISODE = 10000 # Max steps per episode
SAVE_MODEL_EVERY_N_EPISODES = 50 # Save model checkpoint

total_steps = 0
episode_rewards = []
losses = [] # To store loss values

# Initialize frame stacker
frame_stack = deque(maxlen=STACK_SIZE)

print(f"Starting training for {NUM_EPISODES} episodes...")

for episode in range(NUM_EPISODES):
    state, info = env.reset()
    
    # Initialize frame stack for the new episode
    # The first frame is repeated STACK_SIZE times to form the initial state
    current_stacked_frames, frame_stack = stack_frames(frame_stack, state, is_new_episode=True)
    
    episode_reward = 0
    
    for step in range(MAX_STEPS_PER_EPISODE):
        total_steps += 1

        # Agent chooses action
        action = agent.act(current_stacked_frames)
        
        # Environment takes action
        # We need to skip frames and accumulate reward for the skipped frames
        cumulative_reward = 0
        next_state = None
        done = False
        terminated = False
        truncated = False

        for _ in range(SKIP_FRAMES):
            next_frame, reward, terminated, truncated, info = env.step(action)
            cumulative_reward += reward
            done = terminated or truncated
            if done:
                break
        
        next_state = next_frame # The last frame after skipping
        
        # Preprocess and stack next state
        next_stacked_frames, frame_stack = stack_frames(frame_stack, next_state, is_new_episode=False)
        
        # Store experience in replay buffer
        agent.remember(current_stacked_frames, action, cumulative_reward, next_stacked_frames, done)
        
        # Update current state
        current_stacked_frames = next_stacked_frames
        episode_reward += cumulative_reward
        
        # Perform a training step (replay)
        if total_steps > TRAINING_START_STEP: # Start training after buffer has enough experiences
            loss = agent.replay(BATCH_SIZE)
            if loss > 0 : # only append if actual training happened
                losses.append(loss)
        
        # Update target network
        if total_steps % TARGET_NETWORK_UPDATE_FREQ == 0 and total_steps > TRAINING_START_STEP:
            agent.update_target_model()
            print(f"Episode {episode+1}, Step {total_steps}: Target network updated.")

        if done:
            break
    
    episode_rewards.append(episode_reward)
    avg_reward = np.mean(episode_rewards[-100:]) # Moving average of last 100 episodes
    
    print(f"Episode: {episode+1}/{NUM_EPISODES}, Steps: {step+1}, Total Steps: {total_steps}, Reward: {episode_reward:.2f}, Avg Reward (last 100): {avg_reward:.2f}, Epsilon: {agent.epsilon:.4f}")
    
    if losses and total_steps > TRAINING_START_STEP: # Print average loss if training has started
         print(f"Avg Loss (last batch if trained): {np.mean(losses[-BATCH_SIZE:]):.4f}")


    # Save model periodically
    if (episode + 1) % SAVE_MODEL_EVERY_N_EPISODES == 0:
        model_save_path = f"dqn_spaceinvaders_episode_{episode+1}.weights.h5"
        agent.save(model_save_path)
        print(f"Model saved to {model_save_path}")

env.close()
print("Training finished.")

# Plotting training progress (rewards)
plt.figure(figsize=(10,5))
plt.plot(episode_rewards)
plt.title('Episode Rewards')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.show()

# Plotting losses if available
if losses:
    plt.figure(figsize=(10,5))
    plt.plot(losses)
    plt.title('Training Loss (per batch)')
    plt.xlabel('Training Batch')
    plt.ylabel('Loss')
    plt.show()

# 4. Testing and Evaluation

In [ ]:
import time # For adding small delays if rendering

def test_agent(env, agent, num_test_episodes=10, model_path=None, render_mode=None):
    if model_path:
        try:
            agent.load(model_path)
            print(f"Model loaded successfully from {model_path}")
        except Exception as e:
            print(f"Error loading model from {model_path}: {e}")
            print("Testing with potentially untrained agent or initial weights.")
    
    agent.epsilon = 0.0  # Turn off exploration for testing
    total_rewards = []
    
    # Re-initialize frame stacker for testing
    test_frame_stack = deque(maxlen=STACK_SIZE)

    for episode in range(num_test_episodes):
        state, info = env.reset(seed=episode) # Use different seed for each test episode for variability
        
        # Initialize frame stack for the new episode
        current_stacked_frames, test_frame_stack = stack_frames(test_frame_stack, state, is_new_episode=True)
        
        episode_reward = 0
        done = False
        
        print(f"Starting Test Episode {episode + 1}/{num_test_episodes}...")
        
        for step in range(MAX_STEPS_PER_EPISODE): # Use MAX_STEPS_PER_EPISODE from training section
            if render_mode:
                env.render()
                # time.sleep(0.01) # Optional: small delay to make rendering viewable

            action = agent.act(current_stacked_frames) # Epsilon is 0, so greedy action
            
            cumulative_reward_test = 0
            next_state_test = None
            
            for _ in range(SKIP_FRAMES): # Apply frame skipping
                next_frame_test, reward_test, terminated_test, truncated_test, info_test = env.step(action)
                cumulative_reward_test += reward_test
                done = terminated_test or truncated_test
                if done:
                    break
            
            next_state_test = next_frame_test

            if not done: # Only stack if not done, to avoid processing terminal frame for next state
                next_stacked_frames_test, test_frame_stack = stack_frames(test_frame_stack, next_state_test, is_new_episode=False)
                current_stacked_frames = next_stacked_frames_test
            else: # if done, the current_stacked_frames does not change for the next iteration (which won't happen)
                pass

            episode_reward += cumulative_reward_test
            
            if done:
                break
        
        total_rewards.append(episode_reward)
        print(f"Test Episode {episode + 1}: Reward = {episode_reward}")
        if render_mode: # Close render window per episode if opened, or manage outside
             pass # env.close() here would close for all episodes if render_mode is "human"

    avg_reward = np.mean(total_rewards)
    print(f"\nAverage Reward over {num_test_episodes} test episodes: {avg_reward:.2f}")
    
    if avg_reward > 20:
        print(f"SUCCESS! Agent achieved an average reward of {avg_reward:.2f}, which is above 20.")
    else:
        print(f"Requirement not met. Agent achieved an average reward of {avg_reward:.2f}, which is not above 20.")
        
    return avg_reward

# --- Running the Test ---
# Note: Ensure the environment `env` and `agent` are already defined and potentially trained from Section 3.
# If you want to test a specific saved model, provide its path.
# For example, if you saved a model at episode 500:
# LATEST_MODEL_PATH = "dqn_spaceinvaders_episode_500.weights.h5" 
# Or, to test the agent currently in memory after training:
LATEST_MODEL_PATH = None # Set to a path like "dqn_spaceinvaders_episode_X.weights.h5" to load a specific model

NUM_TEST_EPISODES = 20 # As per requirement, run enough tests for a stable average

# To render the game during testing (can be slow, requires a display environment):
# RENDER_TEST_MODE = "human" 
RENDER_TEST_MODE = None # Set to "human" to watch the agent play

# Create a new environment instance for testing if you want to set a render_mode or different options
# If RENDER_TEST_MODE is "human", the env from training (if it was headless) needs to be recreated.
if RENDER_TEST_MODE:
    try:
        test_env = gym.make("SpaceInvaders-v0", obs_type="rgb", render_mode=RENDER_TEST_MODE)
    except Exception:
        test_env = gym.make("SpaceInvaders-v4", obs_type="rgb", render_mode=RENDER_TEST_MODE)
else:
    # Use the existing 'env' if no rendering, or create a new one if 'env' was closed or you prefer isolation
    try:
        test_env = gym.make("SpaceInvaders-v0", obs_type="rgb")
    except Exception:
        test_env = gym.make("SpaceInvaders-v4", obs_type="rgb")


print("\n--- Starting Agent Evaluation ---")
# Ensure agent is defined. If this cell is run independently, 
# you might need to redefine the agent or ensure it's loaded.
# Assuming 'agent' is the trained agent from the previous section.
if 'agent' not in globals():
    print("Agent not found. Please ensure the DQN Agent (Section 2) and Training (Section 3) cells have been run, or load a model.")
    # As a fallback, define a new agent instance (it will be untrained unless LATEST_MODEL_PATH is set)
    # ACTION_SIZE should be available from env, or re-get from test_env
    if 'ACTION_SIZE' not in globals():
       ACTION_SIZE = test_env.action_space.n # Get action size from test_env
    agent = DQNAgent(STATE_SHAPE, ACTION_SIZE)


average_test_reward = test_agent(test_env, agent, num_test_episodes=NUM_TEST_EPISODES, model_path=LATEST_MODEL_PATH, render_mode=RENDER_TEST_MODE)
    
test_env.close() # Clean up the test environment
print("--- Evaluation Finished ---")